# Explore — Synthetic PepsiCo Demand

Interactive look at `data/synthetic/demand.parquet` (Task 1 output).

- Regenerate data: `make data`  ·  Rebuild market-wise EDA workbook: `make eda`
- Independent learning project; **synthetic** data (not real PepsiCo).

In [ ]:
%matplotlib inline
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# find the dataset regardless of where JupyterLab was launched
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data' / 'synthetic' / 'demand.parquet').exists())
df = pd.read_parquet(root / 'data' / 'synthetic' / 'demand.parquet')
df['date'] = pd.to_datetime(df['date'])
print(df.shape)
df.head()

## Overview & data quality

In [ ]:
df.info()
print('\nmissing values per column:')
print(df.isna().sum())
print('\ndate range:', df['date'].min().date(), '->', df['date'].max().date())
df['units'].describe()

## Market-wise summary

In [ ]:
summary = (df.groupby('market')
             .agg(rows=('units', 'size'),
                  total_units=('units', 'sum'),
                  mean_units=('units', 'mean'),
                  zero_pct=('units', lambda s: (s == 0).mean() * 100),
                  stores=('store_id', 'nunique'),
                  skus=('sku_id', 'nunique'),
                  channels=('channel', 'nunique'))
             .sort_values('total_units', ascending=False).round(2))
summary

In [ ]:
summary['total_units'].plot(kind='bar', color='#004b93', title='Total units by market')
plt.tight_layout(); plt.show()

## Channel mix by market
Traditional / General Trade should dominate in India & Pakistan.

In [ ]:
mix = df.groupby(['market', 'channel'])['units'].sum().unstack(fill_value=0)
mix = mix.div(mix.sum(axis=1), axis=0) * 100
mix.round(1)

In [ ]:
mix.plot(kind='barh', stacked=True, figsize=(9, 4), colormap='tab20')
plt.title('Channel share of volume by market (%)')
plt.legend(bbox_to_anchor=(1.01, 1), fontsize=7)
plt.tight_layout(); plt.show()

## Seasonality (weekly + monthly, hemisphere-aware)

In [ ]:
df['dow'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['hemisphere'] = np.where(df['market'] == 'Australia', 'Southern', 'Northern')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
wd = df.groupby('dow')['units'].mean()
wd.index = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
wd.plot(kind='bar', ax=ax[0], color='#004b93', title='Weekly seasonality')
df.groupby(['hemisphere', 'month'])['units'].mean().unstack(0).plot(
    marker='o', ax=ax[1], title='Monthly seasonality (hemisphere flip)')
plt.tight_layout(); plt.show()

## Promotions, holidays & weather

In [ ]:
promo = df.groupby('promo_flag')['units'].mean()
print('promo uplift  :', round(promo.get(1) / promo.get(0), 2), 'x')
hol = df.assign(h=df['holiday'] != '').groupby('h')['units'].mean()
print('holiday uplift:', round(hol.get(True) / hol.get(False), 2), 'x')

bev = df[df['category'] == 'Beverage']
corr = bev.groupby('market').apply(
    lambda s: s.groupby('date').agg(u=('units', 'sum'), t=('temp_c', 'mean')).corr().iloc[0, 1])
print('\nbeverage temp-vs-demand correlation:')
corr.round(2)

## Zoom into one series (SKU × store × day)
Busiest India series, with promotion days marked in red.

In [ ]:
one = df[df['market'] == 'India'].sort_values('date')
key = one.groupby(['store_id', 'sku_id'])['units'].sum().idxmax()  # busiest
s = one[(one['store_id'] == key[0]) & (one['sku_id'] == key[1])]

plt.figure(figsize=(12, 3.5))
plt.plot(s['date'], s['units'], lw=0.8, color='#004b93')
pd_ = s[s['promo_flag'] == 1]
plt.scatter(pd_['date'], pd_['units'], s=12, color='#eb1700', label='promo', zorder=3)
plt.title(f'Daily units — {key[1]} @ {key[0]} (India)')
plt.legend(); plt.tight_layout(); plt.show()

## Reuse the project's EDA helpers
The same functions that build `docs/EDA.xlsx` are importable.

In [ ]:
from cpg_forecast import eda

edf = eda.load_data()
eda.per_market_tables(edf, 'India')['Channel share of volume']